# NimbusPay Ticket Triage — Colab fine-tuning notebook
**Goal:** train a small model to convert support tickets into the exact 8-key JSON schema.

Run top to bottom on a Colab T4. This notebook is deliberately explicit: each section says what it does and why.


## 1. Install packages
Install the Hugging Face training stack and 4-bit quantization support.

In [1]:
!pip -q install -U transformers peft bitsandbytes accelerate datasets scikit-learn


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


## 2. Upload and extract the assignment ZIP
Upload the original `training_assessment.zip` received by email. The notebook reads the provided train/dev/test files directly.

In [2]:
from google.colab import files
uploaded = files.upload()  # Choose training_assessment.zip
ZIP_PATH = next(iter(uploaded))
import zipfile, os, json, pandas as pd, numpy as np
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall("/content/assignment")
ROOT = "/content/assignment/candidate_pack"
print(os.listdir(ROOT))


Saving training_assessment.zip to training_assessment.zip
['data', 'ASSIGNMENT.md', 'SCHEMA.md', 'score.py']


## 3. Audit and clean training labels
The assignment warns that train labels contain errors. This first pass drops malformed JSON and rows where `priority` was corrupted into `prio`. We do not guess what `low` or `medium` should mean. Every dropped row is written to a CSV log.

In [4]:
import csv
EXPECTED = {"category","priority","amount_paise","txn_date","txn_id","channel","language","needs_human"}
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(f"{ROOT}/data/train.jsonl")
dev_rows = read_jsonl(f"{ROOT}/data/dev.jsonl")
test_rows = read_jsonl(f"{ROOT}/data/test_inputs.jsonl")

clean_rows, cleaning_log = [], []
for row in train_rows:
    try:
        label = json.loads(row["messages"][-1]["content"])
    except json.JSONDecodeError:
        cleaning_log.append({"id":row["id"],"action":"drop","reason":"Invalid JSON label"})
        continue
    if set(label) != EXPECTED:
        cleaning_log.append({"id":row["id"],"action":"drop","reason":"Corrupted schema: uses prio instead of priority; value cannot be mapped safely"})
        continue
    if label["priority"] not in {"P1","P2","P3"}:
        cleaning_log.append({"id":row["id"],"action":"drop","reason":"Priority outside allowed values"})
        continue
    clean_rows.append(row)

pd.DataFrame(cleaning_log, columns=["id","action","reason"]).to_csv("cleaning_log.csv", index=False)
print("Original:",len(train_rows)," Retained:",len(clean_rows)," Dropped:",len(cleaning_log))
print(pd.Series([x["reason"] for x in cleaning_log]).value_counts())


Original: 1445  Retained: 1350  Dropped: 95
Corrupted schema: uses prio instead of priority; value cannot be mapped safely    70
Invalid JSON label                                                                25
Name: count, dtype: int64


## 4. Inspect a few examples
Check the ticket text and label before training. The dev set is held out and must never be used for training.

In [5]:
for item in clean_rows[:2]:
    print("ID:", item["id"])
    print("TICKET:", item["messages"][1]["content"][:500])
    print("LABEL:", item["messages"][2]["content"], "\\n")


ID: tr-00001
TICKET: Received: 2026-07-25 (Sat)
Source: app-chat

Hi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.
LABEL: {"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false} \n
ID: tr-00002
TICKET: Received: 2026-06-15 (Mon)
Source: twitter

@NimbusPayCare Where can I reset the passcode?
LABEL: {"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false} \n


## 5. Load the small base model in 4-bit
We use Qwen2.5-0.5B-Instruct: 0.5B parameters, below the assignment's 4B limit. QLoRA keeps the base weights quantized and trains only a small LoRA adapter.

In [6]:
import torch, time, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


## 6. Prepare examples and mask the prompt
The model should learn to generate the assistant JSON, not copy the system/user prompt. We mask prompt tokens with `-100` so they do not contribute to the loss. Sequence length is capped at 1024 to fit a free T4; inspect truncation if results are weak.

In [10]:

MAX_LENGTH = 1024

def encode_row(row):
    msgs = row["messages"]

    # Convert chat messages into text
    prompt_text = tokenizer.apply_chat_template(
        msgs[:2],
        tokenize=False,
        add_generation_prompt=True
    )

    full_text = tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=False
    )

    # Convert text into normal Python lists of token IDs
    prompt_ids = tokenizer(
        prompt_text, add_special_tokens=False
    )["input_ids"]

    full_ids = tokenizer(
        full_text, add_special_tokens=False
    )["input_ids"]

    full_ids = full_ids[:MAX_LENGTH]
    prompt_len = min(len(prompt_ids), len(full_ids))

    labels = [-100] * prompt_len + full_ids[prompt_len:]
    labels = labels[:len(full_ids)]

    pad = MAX_LENGTH - len(full_ids)

    return {
        "input_ids": full_ids + [tokenizer.pad_token_id] * pad,
        "attention_mask": [1] * len(full_ids) + [0] * pad,
        "labels": labels + [-100] * pad
    }


from torch.utils.data import Dataset

class TicketDataset(Dataset):

    def __init__(self, rows):
        self.items = [encode_row(row) for row in rows]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return {
            key: torch.tensor(value, dtype=torch.long)
            for key, value in self.items[idx].items()
        }


train_ds = TicketDataset(clean_rows)
dev_ds = TicketDataset(dev_rows)

print("Training examples:", len(train_ds))
print("Validation examples:", len(dev_ds))


Training examples: 1350
Validation examples: 200


## 7. Train the adapter
This is the initial configuration. For the required controlled experiments, change only one setting per run and record the actual results. Do not report numbers you did not measure.

In [12]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
# Keep a record of runtime and GPU memory for your report.
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

args = TrainingArguments(
    output_dir="/content/nimbuspay-qlora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True
)
class CausalCollator:
    def __call__(self, features):
        return {k:torch.stack([x[k] for x in features]) for k in features[0]}
trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    data_collator=CausalCollator()
)
trainer.train()
train_minutes = (time.time()-start_time)/60
peak_gb = torch.cuda.max_memory_allocated()/1024**3
print(f"Training minutes: {train_minutes:.1f}")
print(f"Peak allocated GPU memory: {peak_gb:.2f} GB")


Epoch,Training Loss,Validation Loss
1,0.030731,0.019104
2,0.016342,0.012147


Training minutes: 24.6
Peak allocated GPU memory: 4.67 GB


## 8. Generate dev predictions and score
Generation uses greedy decoding and max_new_tokens=200, as required. We do not repair model output after generation.

In [17]:

import re
import json
import torch

def generate_output(row):
    prompt = tokenizer.apply_chat_template(
        row["messages"][:2],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


def write_predictions(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            result = {
                "id": row["id"],
                "output": generate_output(row)
            }

            f.write(json.dumps(result, ensure_ascii=False) + "\n")


write_predictions(dev_rows, "/content/predictions_dev.jsonl")

!python {ROOT}/score.py --gold {ROOT}/data/dev.jsonl --pred /content/predictions_dev.jsonl


group                            n   valid   exact  fields
all                            200  100.0%   65.0%   93.8%
category=account_access         19  100.0%  100.0%  100.0%
category=fraud                  15  100.0%   66.7%   92.5%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%   94.4%   99.3%
category=other                  16  100.0%   93.8%   99.2%
category=payment_failed         53  100.0%   47.2%   91.0%
category=refund                 55  100.0%   36.4%   88.6%

per-field accuracy (all rows)
  category        97.5%
  priority        70.0%
  amount_paise    99.0%
  txn_date        96.5%
  txn_id          99.5%
  channel         99.5%
  language       100.0%
  needs_human     88.5%


## 9. Run test inference and save adapter
Run only after you have selected your best configuration. The test set has no labels, so do not use it to tune the model.

In [18]:
write_predictions(test_rows, "/content/predictions_test.jsonl")
model.save_pretrained("/content/adapter")
tokenizer.save_pretrained("/content/tokenizer")
print("Saved adapter and predictions.")


Saved adapter and predictions.


## 10. Download deliverables
Download the adapter folder, predictions, and cleaning log. Save the notebook itself with outputs.

In [19]:
!cd /content && zip -qr adapter.zip adapter
from google.colab import files
files.download("/content/adapter.zip")
files.download("/content/predictions_test.jsonl")
files.download("cleaning_log.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Required experiment plan
Use the same dev set and scorer for each run. Suggested controlled comparisons:
1. Raw data vs cleaned data (keep all other settings fixed).
2. 16-bit LoRA vs 4-bit QLoRA (same model, rank, epochs and learning rate).
3. LoRA rank 8 vs rank 16 (all else fixed).
4. Your own idea, such as sequence length 1024 vs 1536, if GPU memory allows.

Record exact match, mean field accuracy, peak GPU memory and training minutes. A full experiment matrix can take longer than one run; prioritize completing and validating one reproducible run before trying all comparisons.
